# The two-block primitive with the high-pass selector

Companion code for *Types of Bursting with a Two-Block Spiking Primitive*.
This notebook is the entry point. It defines the loop and shows three minimal examples of
what the first-order high-pass selector does. Everything else lives in the other notebooks.

The loop of Section II is

$$y = \mathrm{sat}_k(u + w), \qquad w = H_\mathrm{hp}(s)\,y, \qquad
H_\mathrm{hp}(s) = \frac{\tau s}{\tau s + 1},$$

with the saturation $\mathrm{sat}_k(v) = \tfrac{1}{2}(|kv+1| - |kv-1|)$, as in the paper.
The high-pass has a direct feedthrough, so the output equation $y = \mathrm{sat}_k(u + y - x)$
is implicit and multi-valued for $k > 1$. As in Section II-B, we regularize it with a
low-pass of time constant $\tau_f \ll \tau$, which is what any physical realization does
through parasitics.

| symbol | meaning |
|---|---|
| `y_f` | output of the regularizing low-pass, $\tau_f \dot y_f = -y_f + y$ |
| `x`   | high-pass state, $\tau \dot x = -x + y_f$ |
| `w`   | feedback signal, $w = y_f - x$ |
| `y`   | output of the cell, $y = \mathrm{sat}_k(u + w)$ |

The paper figures are produced elsewhere: Fig. 3 in `simulate_neuron_bursting.ipynb`,
Fig. 4 in `simulate_neuron_typeII.ipynb`, Fig. 5 in `simulate_neuron_elliptic.ipynb`.

In [ ]:
using Plots, LaTeXStrings, DifferentialEquations, Printf, Plots.PlotMeasures

gr(guidefontsize = 14, tickfontsize = 12, legendfontsize = 12, margin = 5Plots.mm, grid = true)
myBlue   = RGBA(131/255, 174/255, 218/255, 1)
myPurple = RGBA(169/255,  90/255, 179/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myGreen  = RGBA(132/255, 195/255, 168/255, 1)
myRed    = RGBA(158/255,   3/255,   8/255, 1)
default(fmt = :png);

## The model

In [ ]:
Base.@kwdef struct HPNeuron
    k::Float64    = 3.0        # sigmoid gain, k > 1 is needed to fire at all
    tau::Float64  = 1.0        # high-pass time constant
    tauf::Float64 = 0.01       # regularizing low-pass, tauf << tau
    u::Function   = t -> 0.0   # external input, at the sigmoid port
end

# algebraic signals from the states X = [y_f, x]
hp_out(y_f, x)        = y_f - x                                         # w
sig_in(y_f, x, p, t)  = hp_out(y_f, x) + p.u(t)                         # v = u + w
sig_out(y_f, x, p, t) = clamp(p.k * sig_in(y_f, x, p, t), -1.0, 1.0)    # y

function rhs!(dX, X, p::HPNeuron, t)
    y_f, x = X
    y = sig_out(y_f, x, p, t)
    dX[1] = (-y_f + y) / p.tauf     # regularizing low-pass
    dX[2] = (-x + y_f) / p.tau      # high-pass state
    return nothing
end

function simulate(p::HPNeuron; tspan = (0.0, 20.0), X0 = [0.1, 0.0],
                  abstol = 1e-9, reltol = 1e-8, dtmax = 0.01, tstops = Float64[])
    prob = ODEProblem(rhs!, X0, tspan, p)
    solve(prob, Rodas5P(); abstol = abstol, reltol = reltol,
          dtmax = dtmax, tstops = tstops)
end

# every node of the loop along a solution
function signals(sol, p::HPNeuron)
    t   = sol.t
    y_f = [s[1] for s in sol.u]
    x   = [s[2] for s in sol.u]
    u   = p.u.(t)
    w   = y_f .- x
    v   = u .+ w
    return (t = t, y_f = y_f, x = x, u = u, w = w, v = v, y = clamp.(p.k .* v, -1.0, 1.0))
end

# edge of the firing band for the saturation, defined for k > 1
rheobase(k) = 1 / k

In [ ]:
# ---------------------------------------------------------------------
#  Plot helper.
#
# The input, the feedback signal and the output, with the edges of the firing band
# drawn on the input panel.
# ---------------------------------------------------------------------

function plot_loop(sol, p::HPNeuron)
    s = signals(sol, p)
    p1 = plot(s.t, s.u; lw = 2, c = :black, ylabel = L"u", legend = false)
    p.k > 1 && hline!(p1, [-rheobase(p.k), rheobase(p.k)]; ls = :dash, c = myRed, label = false)
    p2 = plot(s.t, s.w; lw = 2, c = myBlue,   ylabel = L"w", legend = false)
    p3 = plot(s.t, s.y; lw = 2, c = myPurple, ylabel = L"y", xlabel = L"t/\tau",
              ylims = (-1.05, 1.05), legend = false)
    plot(p1, p2, p3; layout = (3, 1), size = (900, 560), link = :x,
         title = ["k = $(p.k),  tau = $(p.tau)" "" ""], titlefontsize = 11)
end

## Example 1: tonic firing at zero input

The input places the operating point at the center of the sigmoid, where the loop gain is
maximal. The resting point is unstable and the loop settles on a relaxation cycle. This is
the fastest the cell can fire.

In [ ]:
p   = HPNeuron(k = 3.0, tau = 1.0, u = t -> 0.0)
sol = simulate(p; tspan = (0.0, 20.0), X0 = [0.1, 0.0])
plot_loop(sol, p)

## Example 2: the firing band

Firing requires $|u| < u_\mathrm{th}$. Here the bias sits outside the band and the pulse
brings the cell inside it, so the cell fires for exactly as long as the pulse lasts and
falls silent again on release.

In [ ]:
u0, A = 0.8, -0.8                      # bias outside the band, the pulse brings it back to u = 0
p  = HPNeuron(k = 3.0, tau = 1.0, u = t -> u0 + (10.0 <= t <= 30.0 ? A : 0.0))
y0 = clamp(p.k * u0, -1.0, 1.0)        # start from the resting point of the bias
sol = simulate(p; tspan = (0.0, 40.0), X0 = [y0, y0], tstops = [10.0, 30.0])

@printf("k = %.1f  ->  band edge u_th = %.4f\n", p.k, rheobase(p.k))
plot_loop(sol, p)

## Example 3: a slow sweep across the band

A slow input crosses the band twice per period. The rate vanishes at both edges, which is
the parabolic profile of Section IV-B. The endogenous version, where the sweep is produced
by a second copy of the primitive rather than imposed from outside, is in
`simulate_neuron_bursting.ipynb`.

In [ ]:
p   = HPNeuron(k = 3.0, tau = 1.0, u = t -> 0.9 * sin(2pi * t / 200.0))
sol = simulate(p; tspan = (0.0, 200.0), X0 = [0.01, 0.0], dtmax = 0.02)
plot_loop(sol, p)